<a href="https://colab.research.google.com/github/rameshjayamani-oss/GenAI_Assignment/blob/main/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

# Original data
original_loan_data = {
    'Age': [28, 45, 35, 50, 30, 42, 26, 48, 38, 55],
    'AnnualIncome(lakhs)': [6.5, 12, 8, 15, 7, 10, 5.5, 14, 9, 16],
    'CreditScore(300-900)': [720, 680, 750, 640, 710, 660, 730, 650, 700, 620],
    'LoanAmount(lakhs)': [5, 10, 6, 12, 5, 9, 4, 11, 7, 13],
    'LoanTerm(years)': [5, 10, 7, 15, 5, 10, 4, 12, 8, 15],
    'EmploymentType': ['Salaried', 'Self-Employed', 'Salaried', 'Self-Employed', 'Salaried',
                       'Salaried', 'Salaried', 'Self-Employed', 'Salaried', 'Self-Employed'],
    'loan(yes/no)': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
}

original_loan_df = pd.DataFrame(original_loan_data)

np.random.seed(42)

synthetic_loan_records = []

for _ in range(40):

    age = np.random.randint(25, 61)

    # Income loosely correlated with age plus noise
    annual_income = round(np.clip(np.random.normal(0.25 * age, 3), 4, 17), 2)

    # Credit score centered around 680-720 with randomness
    credit_score = int(np.clip(np.random.normal(700, 50), 300, 900))

    # Employment type: roughly 60% Salaried, 40% Self-Employed (based on your data)
    employment_type = np.random.choice(['Salaried', 'Self-Employed'], p=[0.6, 0.4])

    # Loan amount depends on income and credit score (better credit = higher loan)
    loan_amount_base = annual_income * 0.6
    loan_amount_noise = np.random.normal(0, 1)
    loan_amount = round(np.clip(loan_amount_base + loan_amount_noise, 3, 15), 2)

    # Loan term 4-15 years, more likely longer if loan amount higher
    if loan_amount > 10:
        loan_term = np.random.randint(8, 16)
    else:
        loan_term = np.random.randint(4, 11)

    # Loan approval (1 or 0), more likely if better credit score and salaried
    loan_prob = 0.3
    if credit_score > 700:
        loan_prob += 0.4
    if employment_type == 'Salaried':
        loan_prob += 0.2

    loan = np.random.choice([0,1], p=[1-loan_prob, loan_prob])

    synthetic_loan_records.append([
        age,
        annual_income,
        credit_score,
        loan_amount,
        loan_term,
        employment_type,
        loan
    ])

synthetic_loan_df = pd.DataFrame(
    synthetic_loan_records,
    columns=['Age', 'AnnualIncome(lakhs)', 'CreditScore(300-900)', 'LoanAmount(lakhs)', 'LoanTerm(years)', 'EmploymentType', 'loan(yes/no)']
)

# Combine original and synthetic
final_loan_df = pd.concat([original_loan_df, synthetic_loan_df], ignore_index=True)

final_loan_df.head(20)


,Age,AnnualIncome(lakhs),CreditScore(300-900),LoanAmount(lakhs),LoanTerm(years),EmploymentType_Self-Employed,loan(yes/no)
0,28,6.50,720,5.00,5,Salaried,0
1,45,12.00,680,10.00,10,Self-Employed,1
2,35,8.00,750,6.00,7,Salaried,0
3,50,15.00,640,12.00,15,Self-Employed,1
4,30,7.00,710,5.00,5,Salaried,0
5,42,10.00,660,9.00,10,Salaried,1
6,26,5.50,730,4.00,4,Salaried,0
7,48,14.00,650,11.00,12,Self-Employed,1
8,38,9.00,700,7.00,8,Salaried,0
9,55,16.00,620,13.00,15,Self-Employed,1


In [11]:
final_loan_df['EmploymentType'] = final_loan_df['EmploymentType'].map({'Salaried': 1, 'Self-Employed': 0})
final_loan_df['EmploymentType'] = final_loan_df['EmploymentType'].str.strip().str.lower()
final_loan_df['EmploymentType'] = final_loan_df['EmploymentType'].map({
    'salaried': 1,
    'self-employed': 0,
    'self': 0
})

final_loan_df

,Age,AnnualIncome(lakhs),CreditScore(300-900),LoanAmount(lakhs),LoanTerm(years),EmploymentType_Self-Employed,loan(yes/no)
0,28,6.50,720,5.00,5,NaN,0
1,45,12.00,680,10.00,10,NaN,1
2,35,8.00,750,6.00,7,NaN,0
3,50,15.00,640,12.00,15,NaN,1
4,30,7.00,710,5.00,5,NaN,0
5,42,10.00,660,9.00,10,NaN,1
6,26,5.50,730,4.00,4,NaN,0
7,48,14.00,650,11.00,12,NaN,1
8,38,9.00,700,7.00,8,NaN,0
9,55,16.00,620,13.00,15,NaN,1


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
features = final_loan_df[['Age', 'AnnualIncome(lakhs)', 'CreditScore(300-900)', 'LoanAmount(lakhs)', 'LoanTerm(years)', 'EmploymentType_Self-Employed']]
target = final_loan_df['loan(yes/no)']
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

feature_train, feature_test, target_train, target_test = train_test_split(
    features_scaled,
    target,
    test_size=0.3,
    random_state=827
)
kModel = KNeighborsClassifier(n_neighbors=3)
kModel.fit(feature_train, target_train)


ValueError: could not convert string to float: 'Salaried'